# 08. 데이터·모델 개선과 기여 분리

개선 실험은 학습 셀 수를 4,096개로 맞춘 별도 조건이며, 전체 개발 평가 세포 63,036개를 유지합니다. A-series의 12,000개 학습 조건과 구분합니다. 기본 동작은 실제 저장 결과와 작업 상태를 읽는 것입니다. 새 학습은 마지막의 선택 옵션을 켰을 때만 Slurm에 제출합니다.


In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
REPORT = ROOT / 'reports/research_phase2'
protocol = json.loads((ROOT / 'configs/research/hypotheses_v1.json').read_text())
print(protocol['protocol'])


## 데이터를 바꾼 첫 대조

D1은 donor-balanced, D2는 donor×class-balanced sampling입니다. 같은 학습 셀 수를 유지하고, validation/evaluation ID 및 C/class_weight를 고정했습니다. 두 변경은 첫 fold에서 모두 감소했습니다. D1은 후속 기여 분리를 위한 후보이며 확인된 개선이라는 뜻이 아닙니다.


In [ ]:
display(pd.read_csv(REPORT / 'sampling_screen/metrics.tsv', sep='\t'))
print(json.loads((REPORT / 'sampling_screen/audit.json').read_text())['checks'])


## 같은 분류 head로 모델 변경 비교

M01은 동결 encoder + 768→128→6 NN head, M02는 사용되는 마지막 encoder block 9·10만 미세조정, M03은 block 0–10의 query/value 22개 모듈에 rank8 LoRA를 적용합니다. CLS pooling, V2 tokenizer/4096 제한, head 초기화와 optimizer 정책을 맞췄습니다. encoder dropout은 모두 껐으며 학습 parameter와 실제 update 수를 기록했습니다.

LoRA v1은 초기 adapter seed가 명시적으로 고정되지 않아 보존하되 선택에서 제외했습니다. 같은 hyperparameter의 수정 실행 v2를 사용합니다. 후보 선택은 첫 fold의 inner-validation 점수만 읽습니다.


In [ ]:
rows = []
for name in ['M01_full4096_fold0_seed42_v1', 'M02_full4096_fold0_seed42_v1', 'M03_full4096_fold0_seed42_v2']:
    j = json.loads((REPORT / 'runs' / name / 'run.json').read_text())
    rows.append(dict(run_id=name, validation_F1=j['inner_validation_macro_f1'], development_outer_F1=j['metrics']['macro_f1'], encoder_parameters=j['trainable_encoder_parameters'], head_parameters=j['trainable_head_parameters'], epochs=j['chosen_epochs'], reload_max_difference=j['save_reload_probability_max_abs_difference']))
display(pd.DataFrame(rows))
selection = json.loads((ROOT / 'configs/research/confirmation_selection_v2.json').read_text())
print('frozen adapted candidate:', selection['selected_model'])
print('selection rule:', selection['selection_rule'])


## 4개 fold·3개 seed의 동결 대조

M0는 위 M01과 같은 동결 NN head입니다. 각 seed에서 평가된 4개 donor fold의 예측을 합쳐 F1을 계산한 뒤 seed 평균·표준편차를 봅니다. CI는 독립 단위인 8명의 donor를 재표집하여 F1을 다시 계산합니다. seed를 독립 환자로 취급하지 않고, cell bootstrap도 사용하지 않습니다. 학습 fold가 겹치고 donor 수가 작다는 한계가 있습니다.


In [ ]:
display(pd.read_csv(REPORT / 'factorial_frozen_controls/pooled_mean_std.tsv', sep='\t'))
display(pd.read_csv(REPORT / 'factorial_frozen_controls/paired_effects.tsv', sep='\t'))


## 2×2 전체 결과와 실제 작업 상태

DM00=(D0,동결 NN), DM10=(D1,동결 NN), DM01=(D0,선택된 적응 모델), DM11=(D1,선택된 적응 모델)입니다. Interaction은 DM11−DM10−DM01+DM00으로 정의합니다. 전체 결과는 각 조건의 모든 4개 fold·3개 seed가 실제로 끝났을 때만 표시합니다.


In [ ]:
path = REPORT / 'factorial_confirmation/paired_effects.tsv'
if path.exists():
    display(pd.read_csv(path, sep='\t'))
else:
    print('전체 적응 모델 확인이 아직 완료되지 않았습니다.')
job = REPORT / 'adapted_confirmation_slurm_job_id.txt'
if job.exists():
    job_id = job.read_text().strip().split(';')[0]
    print('Slurm array:', job_id)
    result = subprocess.run(['squeue', '-j', job_id, '-o', '%.18i %.28j %.8T %.10M %R'], text=True, capture_output=True)
    print(result.stdout if result.returncode == 0 else result.stderr)


## 외부 cohort의 사전 고정 범위

GSE151530은 25명·32개 HCC 표본이며, 모델·데이터 선택을 고정한 뒤 최종 평가합니다. 원 저자 라벨에는 hepatocyte가 따로 없어서 malignant/unclassified를 Unknown으로 유지합니다. fixed-six 주 점수와 평가 가능한 다섯 lineage의 보조 점수 및 라벨 coverage를 함께 사전 정의했습니다. 이 보조 점수로 hepatocyte 전이를 검증했다고 주장하지 않습니다.


In [ ]:
external = json.loads((REPORT / 'external_metadata/contract.json').read_text())
print('prepared cohort:', external['n_donors'], external['n_samples'], external['n_cells'])
print('label support:', external['label_support'])
print('preparation contract status:', external['status'])
print('primary metric:', external['primary_metric'])


## 선택적인 한 fold 재실행

아래 옵션을 켜면 새 run ID로 동결 NN 조건 하나를 Slurm에 제출합니다. 제출 결과는 학습 완료와 다릅니다. 장기 학습은 서버 안내에 따라 Slurm에서 실행하며 기존 결과를 덮어쓰지 않습니다.


In [ ]:
SUBMIT_ONE_FOLD = False
if SUBMIT_ONE_FOLD:
    from datetime import datetime, timezone
    run_id = 'tutorial08_frozen_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    result = subprocess.run(['sbatch', '--parsable', str(ROOT / 'slurm/improvement_single.sh'), '--model', 'frozen_nn', '--fold', '0', '--seed', '42', '--run-id', run_id], cwd=ROOT, check=True, text=True, capture_output=True)
    print('submitted job:', result.stdout.strip(), 'run:', run_id)
else:
    print('저장 결과 읽기 모드: 새 학습을 제출하지 않았습니다.')
